In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import silhouette_score, silhouette_samples

# Run in Databricks
df_user_behavior = spark.table('workspace.spotify.spotify_user_behavior').toPandas()

CLUSTER_FEATURES_V1 = [
    'daily_listening_minutes',  # usage intensity
    'sessions_per_day',         # frequency
    'days_active_last_30',      # consistency
    'avg_session_minutes',      # depth
    'skip_rate',                # content friction
    'liked_songs_pct',          # positive engagement
    'ads_skipped_pct'           # monetization friction
]

In [0]:
base = df_user_behavior[['user_id'] + CLUSTER_FEATURES_V1].copy()

In [0]:
k =4

In [0]:
X1 = base[CLUSTER_FEATURES_V1].copy()

scaler = MinMaxScaler()
X3_scaled = scaler.fit_transform(X1)
X3 = pd.DataFrame(
    X3_scaled,
    columns = CLUSTER_FEATURES_V1,
    index = base.index
)

km3 = KMeans(
    n_clusters = k,
    random_state = 42,
    n_init = 20,   # more stable than sklearn default (often 10)
    max_iter = 300
)

In [0]:
labels3 = km3.fit_predict(X3)
inertia3 = km3.inertia_

sil_avg_3= silhouette_score(X3, labels3)
base_iter3 = base.copy()
base_iter3['cluster']= labels3
print(f'Inertia: {round(inertia3,2)}')
print(f' Silhoutte: {round(sil_avg_3,2)}')
sizes3 = base_iter3['cluster'].value_counts().sort_index()
sizes3

In [0]:
means3= (
    base_iter3.groupby('cluster')[CLUSTER_FEATURES_V1].mean().round(3))

In [0]:
profile3 = means3.copy()
profile3.insert(0, 'cluster_size', sizes3)
display(profile3)

In [0]:
import matplotlib.pyplot as plt
plt.figure(figsize = (6,4))

base_iter3['cluster'].value_counts().sort_index().plot(kind= 'bar')
plt.title('Cluster size - Iteration 3 (MinMaxScaler)', color = 'red')
plt.xlabel('Cluster')
plt.ylabel('Users')
plt.show()

In [0]:
# scatter plot (No PCA)

# intensity vs skip
import seaborn as sns

plt.figure(figsize= (6,5))
sns.scatterplot(
    data = base_iter3,
    x = 'daily_listening_minutes',
    y = 'skip_rate',
    hue = 'cluster',
    palette = 'tab10'
)

plt.title('Iteration -3 (MinMaxScaler) Intensity vs skip rate', color = 'red')
plt.xlabel('Daily listening minutes')
plt.ylabel('Skip rate')
plt.show()

In [0]:
# frequency vs Session depth

plt.figure(figsize= (6,5))

sns.scatterplot(
    data = base_iter3,
    x = 'sessions_per_day',
    y= 'avg_session_minutes',
    hue = 'cluster',
    palette = 'tab10'
)

plt.title('Iteration - 3 (MinMaxScaler) Session per day Vs Avg session minutes', color = 'red')
plt.xlabel('Sessions in a day')
plt.ylabel(' Avg session minutes')
plt.tight_layout()
plt.show()

In [0]:
# consistance vs Ad friction

plt.figure(figsize= (6,5))

sns.scatterplot(
    data = base_iter3,
    x = 'days_active_last_30',
    y= 'ads_skipped_pct',
    hue = 'cluster',
    palette = 'tab10'
)

plt.title('iteration - 3 (MinMaxScaler) days_active_last_30 VS ads_skipped_pct', color = 'red')
plt.legend(bbox_to_anchor = (1.02, 1))
plt.tight_layout()
plt.show()

In [0]:
# silhouette Plot

sil_values_3 = silhouette_samples(X3, labels3)
sil_avg_3 = silhouette_score(X3, labels3)
plt.figure(figsize= (7,5))

y_lower = 0

vals0 = sil_values_3[labels3 == 0]
vals0.sort()
size0 = vals0.shape[0]

y_upper = y_lower + size0

plt.fill_betweenx(
    np.arange(y_lower, y_upper),    # vertical span
    0,                              # left boundary (silhoutte = 0)
    vals0,                          # right boundary (actual silhoutte values)
    alpha = 0.8
)
plt.text(
    -0.05,
    y_lower + 0.5 * size0,  # slightly left of zero
    'Cluster 0'             # vertically centered
)

y_lower = y_upper

# Cluster 1
vals1 = sil_values_3[labels3==1]
vals1.sort()
size1 = vals1.shape[0]

y_upper = y_lower + size1

plt.fill_betweenx(
    np.arange(y_lower, y_upper),
    0,
    vals1,
    alpha = 0.8
    )
plt.text(-0.05,
         y_lower + 0.5 * size1, 'Cluster 1')

y_lower = y_upper

# cluster 2
vals2 = sil_values_3[labels3 == 2]
vals2.sort()

size2 = vals2.shape[0]
y_upper = y_lower + size2
plt.fill_betweenx(np.arange(y_lower, y_upper), 0, vals2, alpha = 0.8)
plt.text(-0.05, y_lower + 0.5 * size2, 'Cluster 2')

y_lower = y_upper

# cluster 3
vals3 = sil_values_3[labels3 == 3]
vals3.sort()
size3 = vals3.shape[0]
y_upper = y_lower + size3
plt.fill_betweenx(np.arange(y_lower, y_upper), 0, vals3, alpha = 0.8)
plt.text(-0.05, y_lower + 0.5 * size3, 'Cluster 3')

y_lower = y_upper

plt.axvline(
    x= sil_avg_3,
    color = 'red',
    linestyle = '--',
    label =f"Avg silhouette = {sil_avg_3:.2f}"
)

plt.title('silhouette plot (k = 4) - iteration 3 MinMaxScaler', color = 'red')
plt.xlabel('silhouette coefficient', color = 'purple')
plt.ylabel('Sample index (grouped by cluster)', color = 'purple')
plt.legend()
plt.tight_layout()
plt.show